# 01 · Dataset EDA: MultiWOZ (original data) → yes/no items → dialogues

This notebook characterises the original data and shows how each design decision follows from it.
It is the evidence behind the *Data* section and Appendix A of the write-up (`reports/writeup/`).

1. **MultiWOZ 2.2 dialogue corpus** – what real users say (used for the wording of pushback turns).
2. **MultiWOZ database** – the 222 hotel / restaurant / attraction records we use as evidence.
3. **Attribute choice** – which attributes can become clean yes/no questions, and why the others are dropped.
4. **Final item set** – counts, yes/no balance and the pilot / main split, built by `sycoattn.data.multiwoz`.
5. **Dialogues** – one example per condition (A, B3–B5, C, D, control) and segment lengths.

Data: `data/raw/multiwoz/` (`*_db.json` and `MultiWOZ_2.2/{train,dev,test}`). Run `python scripts/prepare_data.py` once to download the DB files.

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# run from the project root wherever the notebook is opened
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from sycoattn.data.multiwoz import CATEGORICAL, build_multiwoz_items, load_db
from sycoattn.dialogue import build_dialogue

DB_DIR = Path("data/raw/multiwoz")
DIALOGUE_DIR = DB_DIR / "MultiWOZ_2.2"
DOMAINS = ["hotel", "restaurant", "attraction"]
pd.set_option("display.max_colwidth", 120)
print("project root:", ROOT)

## 1. MultiWOZ 2.2 dialogue corpus

MultiWOZ (Budzianowski et al., 2018; version 2.2: Zang et al., 2020) is a corpus of human–human, task-oriented dialogues
between a tourist and a clerk in Cambridge (booking hotels, restaurants, trains, ...). We do **not** reuse its dialogues as
experiment inputs; we use the corpus to (a) describe the domain and (b) borrow how real users phrase doubt and disagreement.

In [ ]:
dialogues = []
for split in ["train", "dev", "test"]:
    for f in sorted((DIALOGUE_DIR / split).glob("dialogues_*.json")):
        for d in json.loads(f.read_text()):
            d["_split"] = split
            dialogues.append(d)

n_turns = [len(d["turns"]) for d in dialogues]
user_words = [len(t["utterance"].split()) for d in dialogues for t in d["turns"] if t["speaker"] == "USER"]
pd.DataFrame({
    "dialogues": pd.Series(Counter(d["_split"] for d in dialogues)),
}).T.assign(total=len(dialogues))

In [ ]:
print(f"turns per dialogue: mean {np.mean(n_turns):.1f}, median {np.median(n_turns):.0f}, min {min(n_turns)}, max {max(n_turns)}")
print(f"words per user turn: mean {np.mean(user_words):.1f}, median {np.median(user_words):.0f}")
services = Counter(s for d in dialogues for s in d["services"])
single = sum(len(d["services"]) == 1 for d in dialogues)
print(f"single-domain dialogues: {single} ({100 * single / len(dialogues):.1f}%)")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(n_turns, bins=range(0, max(n_turns) + 2, 2), edgecolor="black")
ax[0].set(xlabel="turns per dialogue", ylabel="dialogues", title="Dialogue length")
s = pd.Series(services).sort_values()
ax[1].barh(s.index, s.values)
ax[1].set(xlabel="dialogues", title="Domains (services)")
plt.tight_layout()

**Real pushback phrasing.** We search user turns for expressions of doubt or disagreement. Most matches are clarifications
("No, I meant ...") rather than disagreement with a fact, so real MultiWOZ users rarely push back hard. We therefore write our
own pushback turns, but keep them short and casual and reuse the real phrasings ("are you sure?", "I thought ...").

In [ ]:
PATTERNS = ["are you sure", "are you certain", "i thought", "i don'?t think", "i believe", "i'?m pretty sure",
            "that'?s not right", "that'?s wrong", "really\\?"]
hits, examples = Counter(), {}
for d in dialogues:
    for i, t in enumerate(d["turns"]):
        if t["speaker"] != "USER":
            continue
        text = t["utterance"].lower()
        for p in PATTERNS:
            if re.search(p, text):
                hits[p] += 1
                examples.setdefault(p, (d["turns"][i - 1]["utterance"] if i else "", t["utterance"]))
pd.DataFrame([{"pattern": p, "user turns": hits[p], "example (system → user)": f"{examples[p][0][:70]} → {examples[p][1][:90]}"}
              for p in PATTERNS if p in hits]).sort_values("user turns", ascending=False)

## 2. MultiWOZ database (the evidence)

Each domain has a JSON database of real Cambridge places. A record is a flat list of attributes, e.g. a hotel has
`name, type, area, pricerange, stars, parking, internet, address, phone, postcode, location, price, takesbookings`.
We render a subset of fields as a short "record" that serves as the evidence in the dialogue.

In [ ]:
db = {d: load_db(d) for d in DOMAINS}
pd.DataFrame([{"domain": d, "records": len(r), "fields": ", ".join(sorted({k for x in r for k in x}))} for d, r in db.items()])

In [ ]:
# value distribution of every candidate attribute (attributes the original code asked about)
CANDIDATES = {"hotel": ["area", "pricerange", "stars", "type", "parking", "internet"],
              "restaurant": ["area", "pricerange", "food"],
              "attraction": ["area", "type"]}
rows = []
for d, attrs in CANDIDATES.items():
    for a in attrs:
        c = Counter(str(r[a]) for r in db[d] if a in r)
        top, n_top = c.most_common(1)[0]
        rows.append({"domain": d, "attribute": a, "distinct values": len(c),
                     "most common": f"{top} ({100 * n_top / sum(c.values()):.0f}%)",
                     "values": dict(c.most_common(8))})
attr_table = pd.DataFrame(rows)
attr_table

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, (d, a) in zip(axes.flat, [("hotel", "area"), ("hotel", "pricerange"), ("hotel", "stars"),
                                  ("restaurant", "area"), ("restaurant", "pricerange"), ("attraction", "area")]):
    c = pd.Series(Counter(str(r[a]) for r in db[d])).sort_values()
    ax.barh(c.index, c.values)
    ax.set_title(f"{d} · {a}")
fig.suptitle("Attributes kept for questions")
plt.tight_layout()

## 3. Which attributes can become clean yes/no questions?

A good attribute has (i) several distinct, non-overlapping values, so a "No" question can ask about a clearly wrong value,
(ii) a balanced Yes/No label, and (iii) an answer that cannot be guessed from the place's name.

**(a) Boolean attributes are badly imbalanced.** For `parking` and `internet` the question is fixed ("Does it have parking?")
and the label comes straight from the record, so almost every answer is "Yes".

In [ ]:
pd.DataFrame([{"attribute": a, "yes": sum(r[a] in ("yes", "free") for r in db["hotel"]),
               "no": sum(r[a] not in ("yes", "free") for r in db["hotel"])} for a in ["parking", "internet"]]
            ).assign(**{"% yes": lambda t: (100 * t.yes / (t.yes + t.no)).round(1)})

**(b) Overlapping categories.** Restaurant `food` has 23 cuisines that overlap (italian vs european vs modern european,
chinese vs asian oriental). If an Italian restaurant is asked "Does it serve European food?", the database label is "No"
but "Yes" is defensible, so a model that holds its answer under pushback might simply be right. Attraction `type` has the
same problem (college / architecture / museum; gallery labelled museum; a typo "mutliple sports").

In [ ]:
print("restaurant food:", dict(Counter(r["food"] for r in db["restaurant"]).most_common()))
print()
print("attraction type:", dict(Counter(r["type"] for r in db["attraction"]).most_common()))

**(c) The name gives the answer away.** Many names contain the value being asked about, so the model can answer from the
name without reading the relevant line of the record.

In [ ]:
TYPE_WORDS = ["guest house", "guesthouse", "hotel", "b and b", "bed and breakfast", "museum", "college", "theatre", "park",
              "cinema", "pool", "boat", "club", "gallery", "church"]
FOOD_WORDS = ["curry", "pizza", "noodle", "chinese", "india", "thai", "bistro", "tapas", "kebab", "sushi"]
AREAS = {"north", "south", "east", "west", "centre"}

def names_with(domain, words):
    return [r["name"] for r in db[domain] if any(w in r["name"] for w in words)]

leak = pd.DataFrame([
    {"attribute": "hotel type", "names revealing it": len(names_with("hotel", TYPE_WORDS)), "records": len(db["hotel"])},
    {"attribute": "attraction type", "names revealing it": len(names_with("attraction", TYPE_WORDS)), "records": len(db["attraction"])},
    {"attribute": "restaurant food", "names revealing it": len(names_with("restaurant", FOOD_WORDS)), "records": len(db["restaurant"])},
    {"attribute": "area (all domains)", "names revealing it": sum(bool(set(r["name"].split()) & AREAS) for d in DOMAINS for r in db[d]),
     "records": sum(len(r) for r in db.values())},
])
leak

In [ ]:
# the area names that are dropped (one of them is misleading: "village centre" is in the east)
[(d, r["name"], r["area"]) for d in DOMAINS for r in db[d] if set(r["name"].split()) & AREAS]

**(d) Hotel `type` and zero-star hotels.** `type` has only two values (guesthouse / hotel), so every "No" question asks about the
other one, and the evidence header "Hotel record:" contradicts the answer to "Is it a hotel?" for guesthouses. Three hotels have
`stars = 0`, which may mean "unrated" rather than "zero stars"; we do not ask stars questions about them.

**(e) Addresses.** 57 restaurant addresses end in "City Centre", which repeats the `area` value in a second line of the record,
so the answer no longer sits in a single line. Three attraction addresses contain "east road" although their area is centre, so
the record contradicts itself. We treat addresses like names and drop these `area` items.

In [ ]:
for d in DOMAINS:
    hits = [(r["address"], r["area"]) for r in db[d] if set(str(r.get("address", "")).lower().replace(",", " ").split()) & AREAS]
    print(f"{d}: {len(hits)} addresses contain an area word", hits[:4])

**Decision.** Ask only about `area` (all domains), `pricerange` (hotel, restaurant) and `stars` (hotel, excluding 0-star).
Drop `parking`, `internet`, `food`, and both `type` attributes as question targets (they stay in the evidence text as
distractor lines). Drop any item whose name or address contains a value of the asked attribute.

In [ ]:
pd.DataFrame([
    {"domain": "hotel", "kept": "area, pricerange, stars", "dropped": "type, parking, internet"},
    {"domain": "restaurant", "kept": "area, pricerange", "dropped": "food"},
    {"domain": "attraction", "kept": "area", "dropped": "type"},
]).assign(questions=[", ".join(CATEGORICAL[d].values()) for d in DOMAINS])

## 4. Final item set

`build_multiwoz_items` builds one yes/no item per (record, kept attribute). For each (domain, attribute) **exactly half** of the
items are "Yes" (ask the record's true value) and half are "No" (ask a random other value). Records are first split into a
**pilot** set (40 records drawn in proportion to each domain's size; used only to tune templates, the prompt and the parser) and a **main** set
(reported results). The split is by record, so the same place never appears in both.

In [ ]:
items = build_multiwoz_items(split="all")
df = pd.DataFrame([{**it.meta, "answer": it.answer_str, "evidence_words": len(it.evidence.split())} for it in items])
print(f"{len(df)} items from {df['name'].nunique()} records")
pd.crosstab([df.domain, df.attribute], df.split, margins=True)

In [ ]:
pd.crosstab([df.split, df.attribute], df.answer)

In [ ]:
print("evidence length (words):", df.evidence_words.describe()[["mean", "min", "max"]].round(1).to_dict())
it = next(i for i in items if i.meta["split"] == "main")
print("\nExample item\n" + it.evidence + "\nQ: " + it.question + "   gold: " + it.answer_str)
print("meta:", it.meta)

## 5. Dialogues

Each item becomes 7 dialogues of up to 5 user turns. The model's replies are generated during the run; here we only show the
user turns. B2 is identical to the first two turns of A, so it is not run separately.

In [ ]:
item = next(i for i in items if i.meta["split"] == "main" and i.meta["attribute"] == "area")
print("Question:", item.question, "| gold:", item.answer_str,
      "| pushback value:", item.meta["wrong_value"], "| true value:", item.meta["true_value"], "\n")
for cond in ["A", "B3", "B4", "B5", "C", "D", "control"]:
    d = build_dialogue(item, cond)
    print(f"--- {cond}")
    for k, t in enumerate(d.turns, 1):
        text = t.text if t.kind != "question" else "[record] ... Based on this record: " + item.question
        if t.kind == "pushback" and cond == "D":
            text = t.segments[0].text + "  [+ record restated]"
        print(f"  t{k} {t.kind:<8} {text}")

In [ ]:
# words per segment type over all main-set dialogues (attention shares depend on span length)
main = [i for i in items if i.meta["split"] == "main"]
seg = Counter(); n = Counter()
for i in main:
    for cond in ["A", "B5", "C", "D"]:
        for t in build_dialogue(i, cond).turns:
            for s in t.segments:
                seg[s.label] += len(s.text.split()); n[s.label] += 1
pd.DataFrame({"segments": pd.Series(n), "mean words": pd.Series({k: seg[k] / n[k] for k in n}).round(1)})